# 03 — Business-Ready Dataset & KPI Engine

**Project:** Gyakriti — Olist Brazilian E-Commerce Analytics  
**Input:** Analytics Base Table (`data/processed/abt_order_items.csv`)  
**Grain:** One row = One Order Item  

---

## Business Objective

The ABT assembled in notebook 02 is structurally correct but analytically raw.
This notebook transforms it into a **business-ready analytical dataset** by:

1. **Standardising datetime columns** — coerce all timestamp fields to proper `datetime64` so that time-based features are accurate and consistent.
2. **Engineering features** — derive time, delivery, financial, and customer features that map directly to business questions (seasonality, logistics performance, revenue concentration, customer retention).
3. **Building reusable KPI tables** — produce six focused aggregation tables covering executive summary, monthly trends, category performance, seller health, customer behaviour, and geography.  These are the canonical inputs for downstream dashboards and stakeholder reports.
4. **Persisting outputs** — save the enriched dataset and all KPI tables as Parquet for efficient downstream consumption.

> **Audience:** This notebook is written for a Senior Data Analyst audience.  
> Comments explain *why* decisions were made, not what the code does.

---
## 0 — Environment Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# ── Project root & src path ────────────────────────────────────────────────
PROJECT_ROOT = Path("/Users/vishwashpatidar/Project Gyakriti/Gyakriti")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from audit import dataframe_overview, missing_value_report

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 60)

---
## 1 — Load the ABT

We load the CSV produced by notebook 02.  
Timestamp columns are **not** parsed here — that happens deliberately in the next section so we can apply `errors='coerce'` uniformly and document every conversion decision.

In [2]:
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
ABT_PATH = PROCESSED_PATH / "abt_order_items.csv"

abt = pd.read_csv(ABT_PATH, low_memory=False)

print(f"Loaded ABT: {abt.shape[0]:,} rows × {abt.shape[1]} columns")
print(f"Source:     {ABT_PATH}")

Loaded ABT: 112,650 rows × 24 columns
Source:     /Users/vishwashpatidar/Project Gyakriti/Gyakriti/data/processed/abt_order_items.csv


### Quick Sanity Check

Before doing anything, confirm the expected grain (112,650 order items) and check overall missingness against the figures established in notebook 02.

In [3]:
EXPECTED_ROWS = 112_650

assert len(abt) == EXPECTED_ROWS, (
    f"Row count mismatch: expected {EXPECTED_ROWS:,}, got {len(abt):,}.  "
    "Re-run notebook 02 to regenerate the ABT."
)

dataframe_overview(abt)

,rows,columns,total_cells,missing_cells,missing_pct,duplicate_rows,memory_mb
0,112650,24,2703600,7862,0.2908,0,51.2278


In [4]:
# Columns with any missing values — should match notebook 02 post-build audit
missing_value_report(abt, threshold=-1).query("missing_count > 0")

,dtype,missing_count,missing_pct,present_count,present_pct
order_delivered_customer_date,str,2454,2.1784,110196,97.8216
product_category_name_english,str,1627,1.4443,111023,98.5557
product_category_name,str,1603,1.4230,111047,98.5770
order_delivered_carrier_date,str,1194,1.0599,111456,98.9401
review_score,float64,942,0.8362,111708,99.1638
product_weight_g,float64,18,0.0160,112632,99.9840
order_approved_at,str,15,0.0133,112635,99.9867
primary_payment_type,str,3,0.0027,112647,99.9973
payment_installments,float64,3,0.0027,112647,99.9973
total_payment_value,float64,3,0.0027,112647,99.9973


---
## 2 — Datetime Standardisation

All five timestamp columns arrive as `object` (string) after a CSV round-trip.  
We convert them with `errors='coerce'` — malformed values become `NaT` rather than raising.

> **Design decision:** Missing datetimes are intentionally left as `NaT`.  
> Filling them (e.g. with column median) would silently corrupt delivery-time calculations and is never appropriate for timestamp data in a logistics dataset.

In [5]:
TIMESTAMP_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in TIMESTAMP_COLS:
    abt[col] = pd.to_datetime(abt[col], errors="coerce")

# Confirm all five columns are now datetime dtype
abt[TIMESTAMP_COLS].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [6]:
# Verify NaT counts — any new nulls beyond the known missingness are a red flag
nat_counts = abt[TIMESTAMP_COLS].isna().sum().rename("nat_count")
nat_counts.to_frame()

,nat_count
order_purchase_timestamp,0
order_approved_at,15
order_delivered_carrier_date,1194
order_delivered_customer_date,2454
order_estimated_delivery_date,0


---
## 3 — Feature Engineering

Features are grouped into four logical domains: **Time**, **Delivery**, **Financial**, and **Customer**.  
All features are added directly to `abt` to keep a single working DataFrame throughout.

### 3.1 Time Features

All time features are derived from `order_purchase_timestamp` — the canonical event time for the purchase.  
`purchase_cohort` is expressed as `YYYY-MM` to enable month-over-month retention analysis without calendar-day noise.

In [7]:
ts = abt["order_purchase_timestamp"]

abt["purchase_year"]        = ts.dt.year
abt["purchase_quarter"]     = ts.dt.quarter
abt["purchase_month"]       = ts.dt.month
abt["purchase_month_name"]  = ts.dt.month_name()
abt["purchase_week"]        = ts.dt.isocalendar().week.astype("Int64")
abt["purchase_day"]         = ts.dt.day
abt["purchase_day_name"]    = ts.dt.day_name()
abt["purchase_hour"]        = ts.dt.hour
abt["purchase_weekday"]     = ts.dt.weekday          # 0 = Monday, 6 = Sunday
abt["is_weekend"]           = ts.dt.weekday >= 5
abt["purchase_cohort"]      = ts.dt.to_period("M").astype(str)

abt[["purchase_year", "purchase_quarter", "purchase_month", "purchase_month_name",
     "purchase_week", "purchase_day", "purchase_day_name", "purchase_hour",
     "purchase_weekday", "is_weekend", "purchase_cohort"]].head(3)

,purchase_year,purchase_quarter,purchase_month,purchase_month_name,purchase_week,purchase_day,purchase_day_name,purchase_hour,purchase_weekday,is_weekend,purchase_cohort
0,2017,3,9,September,37,13,Wednesday,8,2,False,2017-09
1,2017,2,4,April,17,26,Wednesday,10,2,False,2017-04
2,2018,1,1,January,2,14,Sunday,14,6,True,2018-01


### 3.2 Delivery Features

All durations are expressed in **days** (float) to allow fractional precision and easy comparison.  
Rows with `NaT` in either operand naturally propagate to `NaN` — no masking needed.

| Feature | Formula | Business use |
|---|---|---|
| `approval_time_hours` | approved_at − purchase_timestamp (hours) | Checkout-to-approval SLA |
| `carrier_pickup_days` | carrier_date − approved_at (days) | Seller fulfilment speed |
| `delivery_days` | delivered_customer − purchase_timestamp (days) | End-to-end delivery time |
| `estimated_delivery_days` | estimated − purchase_timestamp (days) | Promised delivery window |
| `delivery_delay_days` | delivered_customer − estimated (days) | Positive = late, negative = early |
| `is_delayed` | delivery_delay_days > 0 | Lateness flag |
| `delivered_before_estimate` | delivered_customer < estimated | Early delivery flag |

In [8]:
SECONDS_PER_HOUR = 3600
SECONDS_PER_DAY  = 86_400

abt["approval_time_hours"] = (
    (abt["order_approved_at"] - abt["order_purchase_timestamp"])
    .dt.total_seconds() / SECONDS_PER_HOUR
)

abt["carrier_pickup_days"] = (
    (abt["order_delivered_carrier_date"] - abt["order_approved_at"])
    .dt.total_seconds() / SECONDS_PER_DAY
)

abt["delivery_days"] = (
    (abt["order_delivered_customer_date"] - abt["order_purchase_timestamp"])
    .dt.total_seconds() / SECONDS_PER_DAY
)

abt["estimated_delivery_days"] = (
    (abt["order_estimated_delivery_date"] - abt["order_purchase_timestamp"])
    .dt.total_seconds() / SECONDS_PER_DAY
)

abt["delivery_delay_days"] = (
    (abt["order_delivered_customer_date"] - abt["order_estimated_delivery_date"])
    .dt.total_seconds() / SECONDS_PER_DAY
)

# Boolean flags — NaN in delay → NaN in flag (nullable boolean)
abt["is_delayed"]               = abt["delivery_delay_days"] > 0
abt["delivered_before_estimate"] = (
    abt["order_delivered_customer_date"] < abt["order_estimated_delivery_date"]
)

delivery_cols = [
    "approval_time_hours", "carrier_pickup_days", "delivery_days",
    "estimated_delivery_days", "delivery_delay_days",
    "is_delayed", "delivered_before_estimate",
]

abt[delivery_cols].describe()

,approval_time_hours,carrier_pickup_days,delivery_days,estimated_delivery_days,delivery_delay_days
count,112635.0000,111441.0000,110196.0000,112650.0000,110196.0000
mean,10.5793,2.8508,12.4726,23.8350,-11.3331
std,22.3226,3.5891,9.4457,8.8873,10.1623
min,0.0000,-171.2190,0.5334,2.0080,-146.0161
25%,0.2164,0.8831,6.7363,18.3700,-16.3210
50%,0.3506,1.8385,10.1843,23.2559,-12.0468
75%,15.1972,3.6410,15.5411,28.4715,-6.4768
max,1450.8664,125.7626,209.6286,155.1355,188.9751


### 3.3 Financial Features

- `total_order_value` — item price + freight at the **order-item** grain.  
  This is distinct from `total_payment_value` (which is an order-level aggregate from payments table and may include voucher offsets).
- `freight_percentage` — freight as a share of item price.  A high value signals logistics cost concentration on low-value items.

In [9]:
abt["total_order_value"] = abt["price"] + abt["freight_value"]

# Guard against zero-price items to avoid division by zero
abt["freight_percentage"] = np.where(
    abt["price"] > 0,
    (abt["freight_value"] / abt["price"]) * 100,
    np.nan,
)

abt[["price", "freight_value", "total_order_value", "freight_percentage"]].describe()

,price,freight_value,total_order_value,freight_percentage
count,112650.0000,112650.0000,112650.0000,112650.0000
mean,120.6537,19.9903,140.6441,32.0864
std,183.6339,15.8064,190.7244,34.9894
min,0.8500,0.0000,6.0800,0.0000
25%,39.9000,13.0800,55.2200,13.4034
50%,74.9900,16.2600,92.3200,23.1356
75%,134.9000,21.1500,157.9375,39.3036
max,6735.0000,409.6800,6929.3100,2623.5294


### 3.4 Customer Features

We use `customer_unique_id` (the true entity) to number each customer's orders chronologically.  
`customer_order_number = 1` means the row belongs to that customer's first-ever order.  
`is_repeat_customer` is `True` for any customer who has placed more than one order — a critical input for retention and LTV analysis.

> **Note:** We sort by `order_purchase_timestamp` before ranking.  
> Ties (same customer, same timestamp) are broken arbitrarily — this is expected for multi-item orders where all items share the same purchase timestamp.

In [10]:
# Rank each customer's orders by purchase timestamp
abt = abt.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"],
    na_position="last"
)

abt["customer_order_number"] = (
    abt.groupby("customer_unique_id")["order_purchase_timestamp"]
    .rank(method="dense", ascending=True)
    .astype("Int64")
)

# A customer is 'repeat' if they have more than one distinct order
customer_order_counts = (
    abt.groupby("customer_unique_id")["order_id"]
    .nunique()
    .rename("_total_orders")
)

abt = abt.merge(customer_order_counts, on="customer_unique_id", how="left")
abt["is_repeat_customer"] = abt["_total_orders"] > 1
abt = abt.drop(columns=["_total_orders"])

# Reset to original index order (purchase timestamp sort may have shuffled rows)
abt = abt.reset_index(drop=True)

print(f"Repeat customers: {abt['is_repeat_customer'].sum():,} order-items "
      f"({abt['is_repeat_customer'].mean() * 100:.1f}% of all items)")
abt[["customer_unique_id", "order_id", "order_purchase_timestamp",
     "customer_order_number", "is_repeat_customer"]].head(6)

Repeat customers: 7,483 order-items (6.6% of all items)


,customer_unique_id,order_id,order_purchase_timestamp,customer_order_number,is_repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27,1,False
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27,1,False
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03,1,False
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41,1,False
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42,1,False
5,0004bd2a26a76fe21f786e4fbd80607f,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16,1,False


---
## 4 — KPI Tables

Each KPI table is scoped to a **single business entity**.  
Aggregations are chosen for meaning — not exhaustiveness.  
All tables are ready for direct use in dashboards or stakeholder reports.

> We restrict `delivery_days` and `delivery_delay_days` aggregations to delivered orders only (`order_status == 'delivered'`) to avoid poisoning logistics KPIs with in-transit or cancelled rows.

In [11]:
# Subset used for delivery-related KPIs — delivered orders only
delivered = abt[abt["order_status"] == "delivered"].copy()

print(f"Total order items : {len(abt):,}")
print(f"Delivered items   : {len(delivered):,} ({len(delivered)/len(abt)*100:.1f}%)")

Total order items : 112,650
Delivered items   : 110,197 (97.8%)


### 4.1 Executive KPIs

Top-line business health at a single glance. Intended for board-level or C-suite dashboards.

In [12]:
executive_kpis = pd.DataFrame({
    "total_orders":              [abt["order_id"].nunique()],
    "total_order_items":         [len(abt)],
    "total_customers":           [abt["customer_unique_id"].nunique()],
    "repeat_customers":          [abt[abt["is_repeat_customer"]]["customer_unique_id"].nunique()],
    "total_sellers":             [abt["seller_id"].nunique()],
    "total_revenue_brl":         [abt["price"].sum().round(2)],
    "total_freight_brl":         [abt["freight_value"].sum().round(2)],
    "total_gmv_brl":             [abt["total_order_value"].sum().round(2)],
    "avg_order_item_value_brl":  [abt["price"].mean().round(2)],
    "avg_freight_value_brl":     [abt["freight_value"].mean().round(2)],
    "avg_freight_pct":           [abt["freight_percentage"].mean().round(2)],
    "avg_review_score":          [abt["review_score"].mean().round(2)],
    "pct_delayed":               [(delivered["is_delayed"].mean() * 100).round(2)],
    "avg_delivery_days":         [delivered["delivery_days"].mean().round(2)],
    "median_delivery_days":      [delivered["delivery_days"].median().round(2)],
    "total_product_categories":  [abt["product_category_name_english"].nunique()],
})

executive_kpis.T.rename(columns={0: "value"})

,value
total_orders,98666.0000
total_order_items,112650.0000
total_customers,95420.0000
repeat_customers,2913.0000
total_sellers,3095.0000
total_revenue_brl,13591643.7000
total_freight_brl,2251909.5400
total_gmv_brl,15843553.2400
avg_order_item_value_brl,120.6500
avg_freight_value_brl,19.9900


### 4.2 Monthly KPIs

Month-by-month trend table for revenue, volume, and customer acquisition.  
Expressed at `YYYY-MM` granularity using `purchase_cohort`.

In [13]:
monthly_kpis = (
    abt.groupby("purchase_cohort", observed=True)
    .agg(
        order_count          = ("order_id",              "nunique"),
        order_item_count     = ("order_id",              "count"),
        unique_customers     = ("customer_unique_id",    "nunique"),
        unique_sellers       = ("seller_id",             "nunique"),
        total_revenue_brl    = ("price",                 "sum"),
        total_freight_brl    = ("freight_value",         "sum"),
        total_gmv_brl        = ("total_order_value",     "sum"),
        avg_item_price_brl   = ("price",                 "mean"),
        avg_review_score     = ("review_score",          "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values("purchase_cohort")
)

print(f"Monthly KPIs: {len(monthly_kpis)} months")
monthly_kpis.head(6)

Monthly KPIs: 24 months


,purchase_cohort,order_count,order_item_count,unique_customers,unique_sellers,total_revenue_brl,total_freight_brl,total_gmv_brl,avg_item_price_brl,avg_review_score
0,2016-09,3,6,3,3,267.3600,87.3900,354.7500,44.5600,1.0000
1,2016-10,308,363,305,143,49507.6600,7301.1800,56808.8400,136.3800,3.6000
2,2016-12,1,1,1,1,10.9000,8.7200,19.6200,10.9000,5.0000
3,2017-01,789,955,755,227,120312.8700,16875.6200,137188.4900,125.9800,4.0600
4,2017-02,1733,1951,1708,427,247303.0200,38977.6000,286280.6200,126.7600,4.0500
5,2017-03,2641,3000,2601,499,374344.3000,57704.2900,432048.5900,124.7800,4.0600


### 4.3 Category KPIs

Performance by product category (English label).  
Useful for assortment decisions, marketing prioritisation, and category-level revenue mix.

In [14]:
category_kpis = (
    abt.groupby("product_category_name_english", observed=True, dropna=False)
    .agg(
        order_item_count     = ("order_id",              "count"),
        unique_orders        = ("order_id",              "nunique"),
        unique_products      = ("product_id",            "nunique"),
        unique_sellers       = ("seller_id",             "nunique"),
        total_revenue_brl    = ("price",                 "sum"),
        avg_item_price_brl   = ("price",                 "mean"),
        min_item_price_brl   = ("price",                 "min"),
        max_item_price_brl   = ("price",                 "max"),
        avg_freight_brl      = ("freight_value",         "mean"),
        avg_freight_pct      = ("freight_percentage",    "mean"),
        avg_review_score     = ("review_score",          "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values("total_revenue_brl", ascending=False)
)

print(f"Category KPIs: {len(category_kpis)} categories")
category_kpis.head(10)

Category KPIs: 72 categories


,product_category_name_english,order_item_count,unique_orders,unique_products,unique_sellers,total_revenue_brl,avg_item_price_brl,min_item_price_brl,max_item_price_brl,avg_freight_brl,avg_freight_pct,avg_review_score
43,health_beauty,9670,8836,2444,492,1258681.3400,130.1600,1.2000,3124.0000,18.8800,30.2100,4.1400
70,watches_gifts,5991,5624,1329,101,1205005.6800,201.1400,8.9900,3999.9000,16.7800,17.0600,4.0200
7,bed_bath_table,11115,9417,3029,196,1036988.6800,93.3000,6.9900,1999.9800,18.4200,28.4300,3.9000
65,sports_leisure,8641,7720,2867,481,988048.9700,114.3400,4.5000,4059.0000,19.5100,29.3900,4.1100
15,computers_accessories,7827,6689,1639,287,911954.3200,116.5100,3.9000,3699.9900,18.8200,29.4400,3.9300
39,furniture_decor,8334,6449,2657,370,729762.4900,87.5600,4.9000,1899.0000,20.7300,34.6700,3.9100
20,cool_stuff,3796,3632,789,267,635290.8500,167.3600,7.0000,3109.9900,22.1400,22.0300,4.1500
49,housewares,6964,5884,2335,468,632248.6600,90.7900,3.0600,6735.0000,20.9900,40.2700,4.0500
5,auto,4235,3897,1900,383,592720.1100,139.9600,3.4900,2258.0000,21.8800,33.1700,4.0600
42,garden_tools,4347,3518,753,237,485256.4600,111.6300,6.3500,3930.0000,22.7700,33.2000,4.0500


### 4.4 Seller KPIs

Per-seller performance covering volume, revenue, delivery speed, and satisfaction.  
The delivery KPIs (`avg_delivery_days`, `delay_rate`) are computed from `delivered` only.

In [15]:
seller_base = (
    abt.groupby("seller_id", observed=True)
    .agg(
        seller_city          = ("seller_city",           "first"),
        seller_state         = ("seller_state",          "first"),
        order_item_count     = ("order_id",              "count"),
        unique_orders        = ("order_id",              "nunique"),
        unique_customers     = ("customer_unique_id",    "nunique"),
        unique_categories    = ("product_category_name_english", "nunique"),
        total_revenue_brl    = ("price",                 "sum"),
        avg_item_price_brl   = ("price",                 "mean"),
        total_freight_brl    = ("freight_value",         "sum"),
        avg_review_score     = ("review_score",          "mean"),
    )
    .round(2)
    .reset_index()
)

seller_delivery = (
    delivered.groupby("seller_id", observed=True)
    .agg(
        avg_delivery_days    = ("delivery_days",         "mean"),
        avg_carrier_pickup   = ("carrier_pickup_days",   "mean"),
        delay_rate           = ("is_delayed",            "mean"),
    )
    .round(4)
    .reset_index()
)

seller_kpis = seller_base.merge(seller_delivery, on="seller_id", how="left")
seller_kpis["delay_rate"] = (seller_kpis["delay_rate"] * 100).round(2)

print(f"Seller KPIs: {len(seller_kpis):,} sellers")
seller_kpis.sort_values("total_revenue_brl", ascending=False).head(10)

Seller KPIs: 3,095 sellers


,seller_id,seller_city,seller_state,order_item_count,unique_orders,unique_customers,unique_categories,total_revenue_brl,avg_item_price_brl,total_freight_brl,avg_review_score,avg_delivery_days,avg_carrier_pickup,delay_rate
857,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1156,1132,1124,10,229472.6300,198.5100,20168.0700,4.1200,15.0053,2.3270,11.5900
1013,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,410,358,349,2,222776.0500,543.3600,13080.6300,4.0800,13.3744,3.1218,4.0000
881,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1987,1806,1792,7,200472.9200,100.8900,35067.0400,3.8000,14.4165,2.3570,10.9800
3024,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,586,585,581,5,194042.0300,331.1300,10042.7000,4.3400,13.3419,2.3385,10.1900
1535,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,1364,982,971,6,187923.8900,137.7700,51612.5500,3.3500,22.3925,11.5962,9.5900
1560,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,340,336,336,8,176431.8700,518.9200,6322.1800,4.2100,11.3351,2.4141,5.5900
2643,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1551,1314,1275,4,160236.5700,103.3100,24955.7500,4.0700,11.1697,2.3006,7.3000
1505,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1171,1160,1155,2,141745.5300,121.0500,20902.8500,4.2400,11.1709,1.5825,5.8900
192,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,1428,915,899,4,138968.5500,97.3200,33892.1400,3.8500,12.0513,3.7184,9.2300
1824,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1499,1287,1282,23,135171.7000,90.1700,25430.9800,4.0500,10.7404,1.6898,8.0800


### 4.5 Customer KPIs

Customer-level aggregation enabling LTV, segmentation, and retention analysis.  
`first_purchase` and `last_purchase` establish tenure and churn windows.

In [16]:
customer_kpis = (
    abt.groupby("customer_unique_id", observed=True)
    .agg(
        customer_city        = ("customer_city",         "first"),
        customer_state       = ("customer_state",        "first"),
        total_orders         = ("order_id",              "nunique"),
        total_items          = ("order_id",              "count"),
        total_spend_brl      = ("price",                 "sum"),
        avg_order_value_brl  = ("total_order_value",     "mean"),
        avg_review_score     = ("review_score",          "mean"),
        is_repeat_customer   = ("is_repeat_customer",    "max"),
        first_purchase       = ("order_purchase_timestamp", "min"),
        last_purchase        = ("order_purchase_timestamp", "max"),
        unique_categories    = ("product_category_name_english", "nunique"),
    )
    .reset_index()
)

customer_kpis["total_spend_brl"]     = customer_kpis["total_spend_brl"].round(2)
customer_kpis["avg_order_value_brl"] = customer_kpis["avg_order_value_brl"].round(2)
customer_kpis["avg_review_score"]    = customer_kpis["avg_review_score"].round(2)

print(f"Customer KPIs: {len(customer_kpis):,} unique customers")
customer_kpis.sort_values("total_spend_brl", ascending=False).head(10)

Customer KPIs: 95,420 unique customers


,customer_unique_id,customer_city,customer_state,total_orders,total_items,total_spend_brl,avg_order_value_brl,avg_review_score,is_repeat_customer,first_purchase,last_purchase,unique_categories
3799,0a0a92112bd4c708ca5fde585afaa872,rio de janeiro,RJ,1,8,13440.0000,1708.0100,1.0000,False,2017-09-29 15:24:52,2017-09-29 15:24:52,1
81388,da122df9eeddfedc1dc1f5349a1a690c,araruama,RJ,2,2,7388.0000,3785.8200,5.0000,True,2017-04-01 15:58:40,2017-04-01 15:58:41,1
44139,763c8b1c9c68a0229c42c9fc6f662b93,vila velha,ES,1,4,7160.0000,1818.7200,1.0000,False,2018-07-15 14:49:44,2018-07-15 14:49:44,1
82230,dc4802a71eae9be1dd28f5d788ceb526,campo grande,MS,1,1,6735.0000,6929.3100,5.0000,False,2017-02-12 20:37:36,2017-02-12 20:37:36,1
26015,459bef486812aa25204be022145caa62,vitoria,ES,1,1,6729.0000,6922.2100,NaN,False,2018-07-25 18:10:17,2018-07-25 18:10:17,1
95131,ff4159b92c40ebe40454e3e6a7c35ed6,marilia,SP,1,1,6499.0000,6726.6600,5.0000,False,2017-05-24 18:14:34,2017-05-24 18:14:34,1
23947,4007669dec559734d6f53e029e360987,divinopolis,MG,1,6,5934.6000,1013.5900,1.0000,False,2017-11-24 11:03:35,2017-11-24 11:03:35,1
89056,eebb5dda148d3893cdaf5b5ca3040ccb,maua,SP,1,1,4690.0000,4764.3400,4.0000,False,2017-04-18 18:50:13,2017-04-18 18:50:13,1
34820,5d0a2980b292d049061542014e8960bf,goiania,GO,1,2,4599.9000,2404.7200,1.0000,False,2018-07-12 12:08:36,2018-07-12 12:08:36,1
27242,48e1ac109decbb87765a3eade6854098,joao pessoa,PB,1,1,4590.0000,4681.7800,5.0000,False,2018-06-22 12:23:19,2018-06-22 12:23:19,1


### 4.6 Geography KPIs

Demand-side (customer) geography at the state level.  
Enables regional revenue maps and logistic cost benchmarks by state.

In [17]:
geography_kpis = (
    abt.groupby("customer_state", observed=True)
    .agg(
        order_item_count     = ("order_id",              "count"),
        unique_orders        = ("order_id",              "nunique"),
        unique_customers     = ("customer_unique_id",    "nunique"),
        unique_cities        = ("customer_city",         "nunique"),
        total_revenue_brl    = ("price",                 "sum"),
        avg_item_price_brl   = ("price",                 "mean"),
        total_freight_brl    = ("freight_value",         "sum"),
        avg_freight_brl      = ("freight_value",         "mean"),
        avg_review_score     = ("review_score",          "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values("total_revenue_brl", ascending=False)
)

print(f"Geography KPIs: {len(geography_kpis)} states")
geography_kpis

Geography KPIs: 27 states


,customer_state,order_item_count,unique_orders,unique_customers,unique_cities,total_revenue_brl,avg_item_price_brl,total_freight_brl,avg_freight_brl,avg_review_score
25,SP,47449,41375,39981,628,5202955.0500,109.6500,718723.0700,15.1500,4.1300
18,RJ,14579,12762,12303,148,1824092.6700,125.1200,305589.3100,20.9600,3.8100
10,MG,13129,11544,11178,744,1585308.0300,120.7500,270853.4600,20.6300,4.0900
22,RS,6235,5432,5249,377,750304.0200,120.3400,135522.7400,21.7400,4.0500
17,PR,5740,4998,4840,363,683083.7600,119.0000,117851.6800,20.5300,4.1100
23,SC,4176,3612,3513,239,520553.3400,124.6500,89660.2600,21.4700,4.0000
4,BA,3799,3358,3257,352,511349.9900,134.6000,100156.6800,26.3600,3.8200
6,DF,2406,2125,2062,6,302603.9400,125.7700,50625.5000,21.0400,4.0000
8,GO,2333,2007,1942,177,294591.9500,126.2700,53114.9800,22.7700,4.0000
7,ES,2256,2025,1956,95,275037.3100,121.9100,49764.6000,22.0600,3.9900


---
## 5 — Save Outputs

All outputs are saved as **Parquet** to `data/processed/`.  
Parquet is chosen over CSV for:

- Schema preservation (datetime and boolean dtypes survive round-trips)
- ~10x smaller file size vs. CSV at equivalent information density
- Columnar storage for fast analytical reads in tools like DuckDB, Spark, and Tableau Hyper

The enriched ABT overwrites the CSV source using a Parquet file with the same base name — the CSV can be regenerated from notebook 02 if needed.

In [18]:
OUTPUTS = {
    "analytics_base_table": abt,
    "executive_kpis":       executive_kpis,
    "monthly_kpis":         monthly_kpis,
    "seller_kpis":          seller_kpis,
    "customer_kpis":        customer_kpis,
    "category_kpis":        category_kpis,
    "geography_kpis":       geography_kpis,
}

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

saved = []
for name, df in OUTPUTS.items():
    path = PROCESSED_PATH / f"{name}.parquet"
    df.to_parquet(path, index=False)
    saved.append({"file": path.name, "rows": len(df), "columns": df.shape[1]})
    
pd.DataFrame(saved)

,file,rows,columns
0,analytics_base_table.parquet,112650,46
1,executive_kpis.parquet,1,16
2,monthly_kpis.parquet,24,10
3,seller_kpis.parquet,3095,14
4,customer_kpis.parquet,95420,12
5,category_kpis.parquet,72,12
6,geography_kpis.parquet,27,10


---
## 6 — Notebook Summary

| Output | Grain | Purpose |
|---|---|---|
| `analytics_base_table.parquet` | Order item | Enriched ABT with all derived features |
| `executive_kpis.parquet` | Business total | Board-level single-row summary |
| `monthly_kpis.parquet` | YYYY-MM | Revenue and volume trends |
| `category_kpis.parquet` | Product category | Assortment and revenue mix |
| `seller_kpis.parquet` | Seller | Supply-side performance & logistics |
| `customer_kpis.parquet` | Customer (unique) | LTV, retention, segmentation |
| `geography_kpis.parquet` | Customer state | Regional demand and freight cost |

**Next step →** Notebook 04 will use these KPI tables to produce the primary analytical charts and executive summary visualisations.